# MACOS: Multi-Agent Compute OS — GRPO Training (v2)

**Hackathon**: Meta x PyTorch OpenEnv x Scaler Grand Finale (April 26, 2026)

Trains **Qwen2.5-1.5B** with **GRPO** (Group Relative Policy Optimization) for OS resource governance with multi-agent deception detection.

| Setting | Value |
|---------|-------|
| Model | Qwen2.5-1.5B-Instruct (4-bit LoRA) |
| Method | GRPO with 4 reward verifiers (v2: improved) |
| Runtime | T4 GPU, ~2-3 hours |
| Dataset | 500 env samples (easy/medium/hard) |
| Epochs | 2 |
| Key fixes | Scaled contextual reward, kill penalty, state-aware reasoning, T4-optimized throughput |

**Before running**: Runtime → Change runtime type → **T4 GPU**

---

## Step 0: GPU Check
Verify GPU is available before wasting time on setup.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "NO GPU DETECTED!\n"
        "Go to: Runtime > Change runtime type > T4 GPU > Save\n"
        "Then: Runtime > Restart and run all"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4 (15.6 GB VRAM)
PyTorch: 2.10.0+cu128
CUDA: 12.8


## Step 1: Upload Project Files
Upload `macos-clean.zip` (126KB) when prompted.

In [ ]:
import os
from google.colab import files

PROJECT_DIR = '/content/macos'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

# Check if files already exist (e.g. from a previous run in same session)
if os.path.exists(os.path.join(PROJECT_DIR, 'env', 'core.py')):
    print("Project files already present, skipping upload.")
else:
    print("Upload macos-clean.zip (126KB)...")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    os.system(f'unzip -o "{zip_name}" -d .')
    os.remove(zip_name)

# Verify
assert os.path.exists('env/core.py'), "env/core.py missing! Upload failed."
assert os.path.exists('env/text_wrapper.py'), "env/text_wrapper.py missing!"
assert os.path.exists('train_grpo.py'), "train_grpo.py missing!"
print("All project files verified.")
os.system('ls -la env/')

Upload macos-clean.zip (126KB)...


Saving macos-clean.zip to macos-clean.zip
All project files verified.


0

## Step 2: Install Dependencies
Installs Unsloth + TRL. Does NOT reinstall PyTorch (Colab has GPU torch already).

In [ ]:
%%capture install_output
# Unsloth MUST be installed before trl/transformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>&1 | tail -3
!pip install "trl>=1.2.0" "transformers>=4.45" "datasets>=3.0" "peft>=0.13" "accelerate>=1.0" 2>&1 | tail -3
# DO NOT install torch/xformers/triton — Colab already has GPU versions
print("Installation complete!")

## Step 3: Verify Imports
Import unsloth first, then verify all libraries.

In [ ]:
# Import unsloth FIRST to get all optimizations
try:
    import unsloth
    HAVE_UNSLOTH = True
    print(f"Unsloth: {unsloth.__version__}")
except ImportError:
    HAVE_UNSLOTH = False
    print("Unsloth: not available (will use standard transformers + peft)")

import torch, trl, transformers
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"TRL: {trl.__version__} | Transformers: {transformers.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: 2026.4.8
PyTorch: 2.10.0+cu128 | CUDA: True
TRL: 1.2.0 | Transformers: 5.5.0
GPU: Tesla T4
VRAM: 15.6 GB


## Step 4: Verify Environment
Quick sanity check that the env works before committing to training.

In [ ]:
import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from env.text_wrapper import observation_to_text, generate_training_dataset, SYSTEM_PROMPT
from env.gym_env import AdaptiveOSEnv # Changed from env.core

# Quick test
env = AdaptiveOSEnv()
obs = env.reset()
text = observation_to_text(obs)
print("=== Sample Observation (first 400 chars) ===")
print(text[:400])
print(f"\nSystem prompt: {len(SYSTEM_PROMPT)} chars")
print("Environment OK!")

=== Sample Observation (first 400 chars) ===
=== SYSTEM STATE (Step 0/30) ===
CPU Usage: 0.0% [UNDERUTILIZED]
Queue Length: 4 processes
Total Cost: $0.00
Deception Rate: -100%

=== PROCESSES ===
 PID | Strategy     | Claimed |  True | Pri | Wait | Critical | Status
-------------------------------------------------------------------------------------
   0 | honest       |      0% |   18% |   4 |    0 |      Yes | normal
   1 | honest       | 

System prompt: 1441 chars
Environment OK!


## Step 5: Generate Training Dataset
500 samples across easy/medium/hard difficulties (~1 min). Quality of rewards matters more than dataset size.

In [ ]:
import time
os.chdir(PROJECT_DIR)

start = time.time()
print("Generating training dataset...")

dataset = generate_training_dataset(
    n_episodes_per_task=10,
    max_samples=500,
)

print(f"Generated {len(dataset)} samples in {time.time()-start:.1f}s")
print(f"Columns: {dataset.column_names}")
print(f"\nDifficulty distribution:")
if 'difficulty' in dataset.column_names:
    from collections import Counter
    counts = Counter(dataset['difficulty'])
    for d, c in sorted(counts.items()):
        print(f"  {d}: {c}")
print(f"\nSample prompt (first 200 chars):")
print(dataset[0]['prompt'][:200])

Generating training dataset...
Generated 500 samples in 1.6s
Columns: ['prompt', 'env_state_json', 'task']

Difficulty distribution:

Sample prompt (first 200 chars):
[{'role': 'system', 'content': "You are MACOS, an intelligent resource governance agent for a multi-tenant compute system.\n\nProcesses in your system may be honest, greedy, liars, panicking, or adversarial.\nDeceptive processes lie about their CPU needs to steal more resources.\n\nAVAILABLE ACTIONS:\n- SCHEDULE: Normal load balancing. Use when CPU is healthy (50-70%) and queue is manageable.\n- KILL: Terminate a process. LAST RESORT ONLY when CPU > 90% and system is critical.\n- PRIORITIZE: Boost a process's priority. Use for starved or SLA-critical processes.\n- THROTTLE: Reduce a process's CPU allocation. Best for deceptive agents overclaiming CPU.\n- DELAY: Postpone a non-critical process. Use when queue is building but CPU is moderate.\n- REALLOCATE: Redistribute resources and accept negotiations. Good for fixing sta

## Step 6: Load Model (4-bit LoRA)
Loads Qwen2.5-1.5B-Instruct with 4-bit quantization. Uses ~4GB VRAM on T4.

In [ ]:
os.chdir(PROJECT_DIR)
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
USE_UNSLOTH = False

# Reduced from 2048 → 1536 to cut memory; prompt(1024)+completion(256)+margin
MAX_SEQ_LEN = 1536

if HAVE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel
        print(f"Loading {MODEL_NAME} with Unsloth (4-bit)...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,
            load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            lora_alpha=16,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",  # Saves ~40% VRAM
        )
        USE_UNSLOTH = True
        print("Loaded with Unsloth!")
    except Exception as e:
        print(f"Unsloth load failed: {e}")
        HAVE_UNSLOTH = False

if not USE_UNSLOTH:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    print("Loading Qwen/Qwen2.5-1.5B-Instruct with transformers + peft...")
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        torch_dtype=torch.float16,         # T4 doesn't support bf16
        device_map="auto",
    )
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    ))
    model.gradient_checkpointing_enable()  # Saves VRAM, allows larger batch
    print("Loaded with transformers + PEFT!")

# Pad token setup (avoids padding warnings)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Max seq length: {MAX_SEQ_LEN}")

Loading unsloth/Qwen2.5-1.5B-Instruct with Unsloth (4-bit)...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loaded with Unsloth!

Trainable: 18,464,768 / 1,036,449,280 (1.78%)
VRAM used: 1.65 GB


## Step 7: Define GRPO Reward Verifiers (v2 — improved)
4 reward functions with key fixes:
- **contextual_reward** scaled 3x (core signal, was underweighted vs formatting)
- **kill penalty** added: KILL in non-critical states gets negative reward
- **soft-action bonus**: THROTTLE/DELAY/REALLOCATE get bonus in moderate states
- **reasoning_reward** is now state-aware (matches keywords to actual state)

In [ ]:
import re, json
os.chdir(PROJECT_DIR)
from env.text_wrapper import parse_llm_response, score_action_in_context

VALID_ACTIONS = {"SCHEDULE", "KILL", "PRIORITIZE", "THROTTLE", "DELAY", "REALLOCATE"}
SOFT_ACTIONS = {"THROTTLE", "DELAY", "REALLOCATE", "PRIORITIZE"}

def _get_text(completion):
    """Extract text from completion (handles both str and message list)."""
    return completion if isinstance(completion, str) else completion[0]["content"]

# Reward 1: Correct XML format (<action>, <target_pid>, <reason>, <think>)
def format_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        text = _get_text(c)
        s = 0.0
        if re.search(r'<action>\s*\w+\s*</action>', text, re.I): s += 0.4
        if re.search(r'<target_pid>\s*\d+\s*</target_pid>', text, re.I): s += 0.3
        if re.search(r'<reason>.*?</reason>', text, re.I | re.S): s += 0.15
        if re.search(r'<think>.*?</think>', text, re.I | re.S): s += 0.15
        rewards.append(s)
    return rewards

# Reward 2: Valid action type + valid PID from process list
def valid_action_reward(completions, **kwargs):
    env_states = kwargs.get("env_state_json", [])
    rewards = []
    for i, c in enumerate(completions):
        text = _get_text(c)
        s = 0.0
        m = re.search(r'<action>\s*(.*?)\s*</action>', text, re.I)
        if m and m.group(1).upper().strip() in VALID_ACTIONS:
            action = m.group(1).upper().strip()
            s += 0.5
            pm = re.search(r'<target_pid>\s*(\d+)\s*</target_pid>', text, re.I)
            if action == "SCHEDULE":
                s += 0.3  # SCHEDULE doesn't need a target
            elif pm and i < len(env_states):
                try:
                    state = json.loads(env_states[i])
                    pids = {p.get("pid", -1) for p in state.get("processes", [])}
                    s += 0.5 if int(pm.group(1)) in pids else 0.1
                except (json.JSONDecodeError, ValueError):
                    s += 0.1
        rewards.append(s)
    return rewards

# Reward 3: Contextual quality — SCALED 3x (CORE RLVR signal)
# Also adds kill penalty and soft-action bonus
def contextual_reward(completions, **kwargs):
    env_states = kwargs.get("env_state_json", [])
    tasks = kwargs.get("task", [])
    rewards = []
    for i, c in enumerate(completions):
        text = _get_text(c)
        try:
            state = json.loads(env_states[i]) if i < len(env_states) else {}
            task = tasks[i] if i < len(tasks) else ""

            action_match = re.search(r'<action>\s*(.*?)\s*</action>', text, re.I)
            action_type = action_match.group(1).upper().strip() if action_match else "SCHEDULE"

            pid_match = re.search(r'<target_pid>\s*(\d+)\s*</target_pid>', text, re.I)
            target_pid = int(pid_match.group(1)) if pid_match else None

            # Base contextual score from text_wrapper [-1, 1]
            base_score = score_action_in_context(action_type, target_pid, state, task)

            # Shift to [0, 2] so all rewards are non-negative, then scale 1.5x
            # This makes contextual reward worth up to 3.0 vs 1.0 for format/valid
            r = (base_score + 1.0) * 1.5

            # Kill penalty: penalize KILL when CPU < 85%
            cpu = state.get("cpu_usage", 50)
            if action_type == "KILL" and cpu < 85:
                r -= 0.5  # Discourage trigger-happy killing
            if action_type == "KILL" and cpu < 60:
                r -= 0.5  # Even stronger penalty for killing in healthy state

            # Soft-action bonus: reward non-destructive actions in moderate states
            if action_type in SOFT_ACTIONS and 40 <= cpu <= 90:
                r += 0.3

            rewards.append(max(0.0, r))
        except (json.JSONDecodeError, IndexError, ValueError):
            rewards.append(0.0)
    return rewards

# Reward 4: State-aware reasoning quality (ported from train_grpo.py)
def reasoning_reward(completions, **kwargs):
    env_states = kwargs.get("env_state_json", [])
    rewards = []
    for i, c in enumerate(completions):
        text = _get_text(c)
        s = 0.0

        think = re.search(r'<think>(.*?)</think>', text, re.I | re.S)
        if not think:
            rewards.append(0.0)
            continue

        thinking = think.group(1).lower()

        # Get state context for relevance checking
        if i < len(env_states) and env_states[i]:
            try:
                state = json.loads(env_states[i])
            except json.JSONDecodeError:
                state = {}
        else:
            state = {}

        cpu = state.get("cpu_usage", 50)
        violations = state.get("violations", {})
        starvation = violations.get("starvation_count", 0)
        processes = state.get("processes", [])
        has_liars = any(p.get("strategy") in ("liar", "adversarial") for p in processes)
        has_starved = any(p.get("wait_time", 0) > 5 for p in processes)

        # Reward mentioning CPU level
        if any(w in thinking for w in ("cpu", "usage", "load", "overload", "utiliz")):
            s += 0.1

        # Reward mentioning deception if deceptive agents exist
        if has_liars and any(w in thinking for w in ("decep", "liar", "lying", "overclaim", "fake", "adversar")):
            s += 0.15

        # Reward mentioning starvation if it's happening
        if (starvation > 0 or has_starved) and any(w in thinking for w in ("starv", "wait", "fair", "priority")):
            s += 0.15

        # Reward mentioning CPU thresholds correctly
        if cpu > 80 and any(w in thinking for w in ("high", "critical", "overload", "danger", "above")):
            s += 0.1
        elif cpu < 40 and any(w in thinking for w in ("low", "under", "idle", "below")):
            s += 0.1

        # Penalize very short or very long reasoning
        word_count = len(thinking.split())
        if word_count < 10:
            s -= 0.1
        elif word_count > 300:
            s -= 0.1

        # Bonus for mentioning specific PIDs
        if re.search(r'pid\s*\d+', thinking):
            s += 0.05

        rewards.append(max(0.0, min(0.5, s)))
    return rewards

reward_funcs = [format_reward, valid_action_reward, contextual_reward, reasoning_reward]

# Verify all 4 reward functions work
test_completion = ["<think>High CPU on process 42, greedy behavior detected</think><action>THROTTLE</action><target_pid>42</target_pid><reason>Reduce resource hog</reason>"]
test_state = json.dumps({"cpu_usage": 95, "processes": [{"pid": 42, "cpu": 80, "strategy": "greedy", "wait_time": 0}], "queue_length": 5, "violations": {}})
test_kwargs = {
    "env_state_json": [test_state],
    "task": ["High-priority task is running slowly"]
}
print("Reward verifiers configured (v2):")
print(f"  {'Function':25s} {'Score':>6s}  Notes")
print(f"  {'-'*25} {'-'*6}  {'-'*30}")
for fn in reward_funcs:
    score = fn(test_completion, **test_kwargs)[0]
    max_s = "3.0" if fn.__name__ == "contextual_reward" else "1.0" if fn.__name__ != "reasoning_reward" else "0.5"
    print(f"  {fn.__name__:25s} {score:6.2f}  (max ~{max_s})")

# Show kill penalty in action
kill_in_healthy = ["<think>Kill process</think><action>KILL</action><target_pid>42</target_pid><reason>Remove it</reason>"]
healthy_state = json.dumps({"cpu_usage": 45, "processes": [{"pid": 42, "cpu": 20, "strategy": "honest", "wait_time": 0}], "queue_length": 2, "violations": {}})
kill_score = contextual_reward(kill_in_healthy, env_state_json=[healthy_state], task=["test"])[0]
print(f"\n  Kill penalty test (CPU=45%): contextual_reward = {kill_score:.2f} (should be low)")

Reward verifiers configured:
  format_reward             = 1.00
  valid_action_reward       = 1.00
  contextual_reward         = 0.20
  reasoning_reward          = 0.20


## Step 8: Train with GRPO (v2 — T4-optimized)
Optimized for ~2-3h on T4: batch=4, grad_accum=2, shorter sequences, fp16.
Key quality gains come from improved reward functions, not more steps.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import glob
os.chdir(PROJECT_DIR)

OUTPUT_DIR = "macos-llm-grpo"

# Check for existing checkpoint to resume from
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f"Found checkpoint: {resume_from} — will resume training!")
else:
    print("Starting fresh training run.")

# ── T4-optimized config ──────────────────────────────────────────────
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    # Duration: 2 epochs × 500 samples ÷ batch 8 = ~125 optimizer steps
    num_train_epochs=2,

    # Throughput: batch=4, grad_accum=2 → effective batch 8
    # Larger micro-batch = better GPU utilization on T4
    per_device_train_batch_size=4,         # was 2 → 4 (fills T4 better)
    gradient_accumulation_steps=2,         # was 4 → 2 (fewer sync points)

    # LR: higher to compensate for fewer optimizer steps
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # GRPO: 2 generations (each extra gen = ~2x inference cost)
    num_generations=2,

    # Precision: T4 = fp16 only (no bf16 support)
    fp16=True,
    bf16=False,

    # Checkpointing: save every 50 optimizer steps
    logging_steps=10,
    save_steps=50,
    save_total_limit=3,

    # Misc
    report_to="none",
    seed=42,
    dataloader_num_workers=2,              # Overlap data loading with compute
    dataloader_pin_memory=True,            # Faster host→GPU transfer
)

# ── Sequence lengths (shorter = faster per step) ─────────────────────
# Prompts are ~600-900 tokens; 1024 covers 99%+ without waste
# Completions: <think>+<action>+<target_pid>+<reason> fits in ~256 tokens
MAX_PROMPT_LEN = 1024                      # was 1536
MAX_COMPLETION_LEN = 256                   # was 384

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=dataset,
    reward_funcs=reward_funcs,
    max_completion_length=MAX_COMPLETION_LEN,
    max_prompt_length=MAX_PROMPT_LEN,
)

# ── Print training plan ──────────────────────────────────────────────
eff_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = len(dataset) // training_args.per_device_train_batch_size
optimizer_steps = (steps_per_epoch * int(training_args.num_train_epochs)) // training_args.gradient_accumulation_steps
print(f"\n{'='*55}")
print(f"  TRAINING CONFIG v2 (T4-optimized)")
print(f"{'='*55}")
print(f"  Dataset:           {len(dataset)} samples")
print(f"  Effective batch:   {eff_batch} (micro={training_args.per_device_train_batch_size} × accum={training_args.gradient_accumulation_steps})")
print(f"  Optimizer steps:   ~{optimizer_steps} ({int(training_args.num_train_epochs)} epochs)")
print(f"  Generations:       {training_args.num_generations}")
print(f"  Learning rate:     {training_args.learning_rate}")
print(f"  Prompt / Completion: {MAX_PROMPT_LEN} / {MAX_COMPLETION_LEN} tokens")
print(f"  Precision:         fp16")
print(f"  Est. time:         ~2-3 hours on T4")
print(f"{'='*55}")
print(f"  Quality gains from: improved reward functions")
print(f"  (kill penalty, 3x contextual weight, state-aware reasoning)")
print(f"{'='*55}\n")

start = time.time()
trainer.train(resume_from_checkpoint=resume_from)
elapsed = time.time() - start

# Save final model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nTraining complete in {elapsed/60:.1f} minutes!")
print(f"Model saved to {OUTPUT_DIR}/")
print(f"GPU peak memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting fresh training run.

Dataset: 500 samples
Batch: 8 (2 x 4 grad_accum)
Estimated steps: ~62
Generations per prompt: 2
Estimated time: ~1.5-2 hours on T4



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attent

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward / mean,rewards / format_reward / std,rewards / valid_action_reward / mean,rewards / valid_action_reward / std,rewards / contextual_reward / mean,rewards / contextual_reward / std,rewards / reasoning_reward / mean,rewards / reasoning_reward / std
10,-0.057654,2.049375,0.768271,135.687500,61.300000,238.600000,0.075000,126.335716,61.300000,212.000000,0.000013,0.876875,0.103289,0.892500,0.196618,-0.087500,0.410534,0.367500,0.450640
20,-0.065602,2.060000,0.729486,139.462500,58.100000,241.300000,0.087500,128.198811,58.100000,215.800000,0.000027,0.875625,0.134988,0.897500,0.210709,-0.094375,0.374558,0.381250,0.458650
30,-0.042119,1.993750,0.901249,146.562500,52.100000,251.300000,0.137500,127.398813,52.100000,205.100000,0.000144,0.836250,0.205739,0.828750,0.311429,-0.046250,0.333448,0.375000,0.446739
40,-0.009093,2.311250,0.846836,157.387500,66.300000,235.000000,0.100000,144.775597,66.300000,221.100000,0.000619,0.880625,0.171720,0.892500,0.244318,0.001875,0.335938,0.536250,0.444682
50,0.010825,2.318750,0.762462,148.112500,72.300000,227.000000,0.087500,138.364883,72.300000,202.700000,0.001729,0.874375,0.146481,0.887500,0.180342,0.050625,0.342863,0.506250,0.445326
60,-0.023089,2.465000,0.905601,167.250000,71.400000,245.500000,0.100000,157.285123,71.400000,234.000000,0.003108,0.893750,0.179725,0.877500,0.236173,0.008750,0.404965,0.685000,0.402832
70,-0.038934,2.680625,0.699051,177.437500,85.700000,250.900000,0.150000,162.780480,85.700000,221.900000,0.003689,0.920625,0.119061,0.911250,0.173652,0.153750,0.281161,0.695000,0.365631
80,-0.031750,2.478125,0.947624,162.312500,76.700000,247.700000,0.162500,144.975005,76.700000,212.300000,0.006111,0.880000,0.210957,0.863750,0.247781,0.041875,0.366675,0.692500,0.401475
90,0.030409,2.463750,0.906420,173.612500,75.900000,251.700000,0.150000,159.084050,75.900000,237.100000,0.005608,0.869375,0.227182,0.797500,0.305347,0.063125,0.335570,0.733750,0.360315
100,-0.045489,2.476875,0.843132,171.287500,95.000000,252.000000,0.175000,153.958574,95.000000,216.000000,0.006342,0.888750,0.196974,0.870000,0.224022,0.008125,0.352698,0.710000,0.384347


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


Training complete in 78.8 minutes!
Model saved to macos-llm-grpo/
GPU peak memory: 3.77 GB


## Step 9: Save Model + Metadata

In [ ]:
import json, datetime
os.chdir(PROJECT_DIR)

metadata = {
    "model_name": "MACOS-GRPO-Qwen2.5-1.5B-v2",
    "base_model": MODEL_NAME,
    "method": "GRPO (Group Relative Policy Optimization) v2",
    "epochs": 2,
    "training_samples": len(dataset),
    "training_time_minutes": round(elapsed / 60, 1),
    "reward_verifiers": [f.__name__ for f in reward_funcs],
    "reward_improvements": [
        "contextual_reward scaled 3x (was 1x)",
        "kill penalty for CPU < 85%",
        "soft-action bonus for moderate states",
        "state-aware reasoning_reward",
    ],
    "lora_r": 16,
    "quantization": "4-bit",
    "timestamp": datetime.datetime.now().isoformat(),
}

with open(f"{OUTPUT_DIR}/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model_metadata.json:")
print(json.dumps(metadata, indent=2))

# List saved files
import os as _os
saved = _os.listdir(OUTPUT_DIR)
print(f"\nFiles in {OUTPUT_DIR}/ ({len(saved)} files):")
for fn in sorted(saved):
    size = _os.path.getsize(f"{OUTPUT_DIR}/{fn}")
    print(f"  {fn:40s} {size/1e6:.1f} MB" if size > 1e6 else f"  {fn}")

Saved model_metadata.json:
{
  "model_name": "MACOS-GRPO-Qwen2.5-1.5B",
  "base_model": "unsloth/Qwen2.5-1.5B-Instruct",
  "method": "GRPO (Group Relative Policy Optimization)",
  "epochs": 1,
  "training_samples": 500,
  "training_time_minutes": 78.8,
  "reward_verifiers": [
    "format_reward",
    "valid_action_reward",
    "contextual_reward",
    "reasoning_reward"
  ],
  "lora_r": 16,
  "quantization": "4-bit",
  "timestamp": "2026-04-24T20:29:35.049470"
}

Files in macos-llm-grpo/ (11 files):
  README.md
  adapter_config.json
  adapter_model.safetensors                73.9 MB
  chat_template.jinja
  checkpoint-100
  checkpoint-125
  checkpoint-50
  model_metadata.json
  tokenizer.json                           11.4 MB
  tokenizer_config.json
  training_args.bin


## Step 10: Test Trained Model

In [ ]:
os.chdir(PROJECT_DIR)
from env.text_wrapper import observation_to_text, parse_llm_response, SYSTEM_PROMPT
from env.gym_env import AdaptiveOSEnv

if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)

env = AdaptiveOSEnv()
obs = env.reset()
text_obs = observation_to_text(obs)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": text_obs},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
with torch.no_grad():
    output = model.generate(inputs, max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True)
response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

print("="*60)
print("TRAINED MODEL TEST (v2)")
print("="*60)
print(f"\nObservation (first 300 chars):\n{text_obs[:300]}...")
print(f"\nModel Response:\n{response}")

parsed = parse_llm_response(response)
print(f"\nParsed Action: {parsed}")

# Quick 5-step episode
print(f"\n{'='*60}")
print("5-Step Episode Test:")
total_r = 0
for step in range(5):
    obs_t = observation_to_text(obs)
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": obs_t}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(inp, max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True)
    resp = tokenizer.decode(out[0][inp.shape[-1]:], skip_special_tokens=True)
    action = parse_llm_response(resp)
    obs, reward, done, info = env.step(action)
    total_r += float(reward.value)
    print(f"  Step {step+1}: action={action.action_type if action and hasattr(action, 'action_type') else '?':12s} pid={action.target_pid if action and hasattr(action, 'target_pid') else '?':>3s} reward={float(reward.value):.3f}")
    if done:
        break

print(f"\nTotal reward over {step+1} steps: {total_r:.3f}")
print(f"Final grade: {info.get('grade', 'N/A')}")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

TRAINED MODEL TEST

Observation (first 300 chars):
=== SYSTEM STATE (Step 0/30) ===
CPU Usage: 0.0% [UNDERUTILIZED]
Queue Length: 4 processes
Total Cost: $0.00
Deception Rate: -100%

=== PROCESSES ===
 PID | Strategy     | Claimed |  True | Pri | Wait | Critical | Status
-------------------------------------------------------------------------------...

Model Response:
<think>
The system is underutilized with only 0.0% CPU usage. The queue length is 4, which suggests there might be some processes that need prioritization to avoid starvation. There are no deception rates or SLA violations, so we can focus on keeping the CPU at a healthy level without any immediate threats. Given the current low utilization, we should consider allowing a bit of inflation to keep the CPU usage around 50-80%, while still monitoring for potential starvation issues.
</think>

<action>KILL</action>
<target_pid>0</target_pid>
<reason>The system is currently underutilized, and we want to ensure it does not wast

AttributeError: 'Action' object has no attribute 'get'

## Step 11: Download Trained Model
Creates a zip with the LoRA adapter + project code for local use.

In [ ]:
import shutil, os, glob
os.chdir(PROJECT_DIR)

zip_name = "macos-trained"
zip_file = f"{zip_name}.zip"

# Check if zip already exists (from previous run)
if os.path.exists(zip_file):
    print(f"Found existing {zip_file} ({os.path.getsize(zip_file)/1e6:.1f} MB)")
    print("This was created from a previous successful training run.")
    from google.colab import files
    print("\nDownloading existing trained model zip...")
    files.download(zip_file)
    print("✅ Download complete! Extract and use with your base model locally.")

elif os.path.exists(OUTPUT_DIR):
    # Directory exists, create new zip
    print(f"Zipping model from: {OUTPUT_DIR}")
    shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
    print(f"Created {zip_file} ({os.path.getsize(zip_file)/1e6:.1f} MB)")

    from google.colab import files
    print("\nDownloading trained model...")
    files.download(zip_file)
    print("✅ Download complete! Extract and use with your base model locally.")

else:
    # Neither zip nor directory exists - error
    print(f"❌ ERROR: Neither {zip_file} nor {OUTPUT_DIR}/ found!")
    print(f"Current directory: {os.getcwd()}")
    print(f"Contents: {os.listdir('.')}")

    # Look for any saved models
    possible_dirs = [d for d in os.listdir('.') if os.path.isdir(d) and ('llm' in d.lower() or 'grpo' in d.lower() or 'checkpoint' in d.lower())]
    if possible_dirs:
        print(f"\nFound possible model directories: {possible_dirs}")
        print("Run the Emergency cell (last cell) to recover from checkpoints.")
    else:
        print("\n⚠️  No model found. Training may not have completed successfully.")
        print("Check step 8 output or try the Emergency recovery cell.")

Zipping model from: macos-llm-grpo
Created macos-trained.zip (382.8 MB)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete! Extract and use with your base model locally.


## Emergency: Save from Checkpoint (if training disconnected)
Run this cell if training was interrupted. It loads the latest checkpoint and saves it as the final model.

In [ ]:
import glob, shutil, os
PROJECT_DIR = "/content/macos"
OUTPUT_DIR = "macos-llm-grpo"
os.chdir(PROJECT_DIR)

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
if checkpoints:
    latest = checkpoints[-1]
    print(f"Found checkpoint: {latest}")
    # Copy checkpoint files to main output dir
    os.makedirs(OUTPUT_DIR, exist_ok=True) # Ensure OUTPUT_DIR exists to copy into
    for f in os.listdir(latest):
        src = os.path.join(latest, f)
        dst = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
    print(f"Copied checkpoint files to {OUTPUT_DIR}/")
    print("You can now run Step 10 (Test) and Step 11 (Download).")
else:
    print("No checkpoints found. Training may not have started.")

No checkpoints found. Training may not have started.
